# 04 — Group Relative Policy Optimization (GRPO) on CPU

**GRPO** is reinforcement learning with a *verifiable reward*: there are no target answers and no preference
pairs. For each prompt the model samples a group of answers, a reward function scores them, and the model moves
towards the answers that scored above their group's average. It is how reasoning models are trained on math and
code, where the answer can be checked.

Same booking task. The reward (`booking_reward` in `scripts/booking_data.py`):

* 1.0 for bare JSON that exactly matches
* otherwise 0.6 × the share of correct fields, + 0.2 if it is bare JSON (no markdown code block)
* 0 if no JSON can be parsed

Partial credit matters: the base model's answers are never exactly right, so without it every answer in a group
would score 0 and there would be nothing to learn from. The job samples 4 answers for each of 4 requests per step
(no vLLM, no KL term so no reference model), for 40 steps. Expected: ~10 minutes, most of it sampling on CPU.

In [ ]:
%pip install -q "sagemaker==3.21.0"

In [ ]:
import sys

sys.path.insert(0, "scripts")
from sm_helpers import Lab

lab = Lab()
CAPACITY = "fallback"   # "on-demand" | "spot" | "fallback" (compared side by side in 01-sft.ipynb)

In [ ]:
import booking_data

train, _ = booking_data.make_splits()
target = train[0]["target"]
for answer in (target, "```json\n" + target + "\n```", target.replace("Sunday", "sunday"), "sorry, no idea"):
    print(f"{booking_data.booking_reward(answer, target):.2f}  {answer!r}")

## Train

In [ ]:
job = lab.submit("grpo", "train_grpo.py", {"steps": 40}, capacity=CAPACITY, requirements="requirements-trl.txt")

In [ ]:
job = globals().get("job") or lab.latest("grpo")   # after a kernel restart
desc = lab.wait(job)

## Results

In [ ]:
m = lab.metrics(job)
print(f"{'':10} {'exact JSON':>10} {'mean reward':>12} " + " ".join(f"{f:>10}" for f in ("name", "party_size", "day", "time")))
for which in ("base", "finetuned"):
    r = m[which]
    print(f"{which:10} {r['exact_match']:10.0%} {r['mean_reward']:12.2f} "
          + " ".join(f"{r['acc_' + f]:10.0%}" for f in ("name", "party_size", "day", "time")))
curve = [p["reward"] for p in m["reward_curve"] if p["reward"] is not None]
print("\nreward per step (training samples):")
print(" ".join(f"{r:.2f}" for r in curve))
ex = m["examples"][0]
print(f"\nbase:      {ex['base']!r}\nfinetuned: {ex['finetuned']!r}\ntarget:    {ex['target']}")

GRPO only reinforces what the model already samples. It quickly drops the markdown code block and fixes the
weekday (both appear among its samples), but rarely discovers the 24-hour time format on its own, so exact match
lags behind SFT. That is the trade-off: RL needs no labelled answers, but it learns slowly what the model does not
already do sometimes. Starting GRPO from an SFT model is the usual recipe.